# Task 060: refellips — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Spectroscopic ellipsometry inversion using refellips

**Paper**: refnx: neutron and X-ray reflectometry analysis in Python
**Repository**: https://github.com/refnx/refnx

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: N/A (parameter estimation)
- **SSIM**: N/A
- **Evaluation**: Spectroscopic ellipsometry thin-film fitting (CC_psi=0.99999996, thickness_error=4.4e-5 nm)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
refellips — Spectroscopic Ellipsometry Inverse Problem
========================================================
Task: Extract thin-film optical constants (n, k, thickness) from
      ellipsometric Ψ/Δ measurements.

Inverse Problem:
    Given Ψ(λ) and Δ(λ) ellipsometric angles, recover the refractive
    index n(λ), extinction coefficient k(λ), and film thickness d.

Forward Model (refellips):
    Uses the transfer-matrix method for stratified media to compute
    Ψ and Δ from layer optical constants and thicknesses.

Inverse Solver:
    Differential Evolution + Least-Squares refinement using
    refellips/refnx analysis framework.

Repo: https://github.com/refnx/refellips
Paper: Lesina et al. (2022), SoftwareX, 20, 101225.

Usage:
    /data/yjh/spectro_env/bin/python refellips_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`cauchy_n`, `si_optical_constants`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. Forward Model (Transfer Matrix)
# ═══════════════════════════════════════════════════════════
def cauchy_n(wavelength_nm, A, B, C):
    """Cauchy dispersion: n(λ) = A + B/λ² + C/λ⁴"""
    lam_um = wavelength_nm / 1000.0
    return A + B / lam_um**2 + C / lam_um**4

def si_optical_constants(wavelength_nm):
    """Approximate Si optical constants (simple dispersion)."""
    # Simplified model based on Palik data
    lam_um = wavelength_nm / 1000.0
    n = N_SI_633 + 0.8 * (0.633 / lam_um - 1)
    k = K_SI_633 * (0.633 / lam_um) ** 2
    return n, k

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `forward_operator`, `load_or_generate_data`

In [ ]:
def forward_operator(params, wavelengths, angle_deg):
    """
    Compute ellipsometric Ψ and Δ for a thin film on Si substrate
    using the transfer matrix method.

    This implements the standard 2×2 transfer matrix for
    ambient/film/substrate with Fresnel coefficients.

    Parameters
    ----------
    params : dict       Film parameters (thickness, Cauchy coeffs).
    wavelengths : array Wavelengths [nm].
    angle_deg : float   Angle of incidence [degrees].

    Returns
    -------
    psi : array  Ψ [degrees]
    delta : array  Δ [degrees]
    """
    theta0 = np.radians(angle_deg)
    n0 = 1.0  # air

    psi = np.zeros(len(wavelengths))
    delta = np.zeros(len(wavelengths))

    for i, lam in enumerate(wavelengths):
        # Film optical constants
        n1 = cauchy_n(lam, params["A"], params["B"], params["C"])
        k1 = params["k_amp"] * (400.0 / lam) ** 2  # Urbach-like absorption
        N1 = n1 + 1j * k1

        # Substrate
        n2, k2 = si_optical_constants(lam)
        N2 = n2 + 1j * k2

        # Snell's law
        sin_theta0 = np.sin(theta0)
        cos_theta0 = np.cos(theta0)
        cos_theta1 = np.sqrt(1 - (n0 * sin_theta0 / N1) ** 2)
        cos_theta2 = np.sqrt(1 - (n0 * sin_theta0 / N2) ** 2)

        # Fresnel coefficients: air→film
        rp01 = (N1 * cos_theta0 - n0 * cos_theta1) / (N1 * cos_theta0 + n0 * cos_theta1)
        rs01 = (n0 * cos_theta0 - N1 * cos_theta1) / (n0 * cos_theta0 + N1 * cos_theta1)

        # Fresnel coefficients: film→substrate
        rp12 = (N2 * cos_theta1 - N1 * cos_theta2) / (N2 * cos_theta1 + N1 * cos_theta2)
        rs12 = (N1 * cos_theta1 - N2 * cos_theta2) / (N1 * cos_theta1 + N2 * cos_theta2)

        # Phase thickness
        beta = 2 * np.pi * params["thickness"] * N1 * cos_theta1 / lam

        # Total reflection coefficients (Airy formula)
        phase = np.exp(-2j * beta)
        Rp = (rp01 + rp12 * phase) / (1 + rp01 * rp12 * phase)
        Rs = (rs01 + rs12 * phase) / (1 + rs01 * rs12 * phase)

        # Ellipsometric ratio
        rho = Rp / Rs
        psi[i] = np.degrees(np.arctan(np.abs(rho)))
        delta[i] = np.degrees(np.angle(rho))

    return psi, delta

# ═══════════════════════════════════════════════════════════
# 3. Data Generation
# ═══════════════════════════════════════════════════════════
def load_or_generate_data():
    """Generate synthetic ellipsometry data."""
    print("[DATA] Generating synthetic ellipsometry data ...")

    wavelengths = np.linspace(WAVELENGTH_MIN, WAVELENGTH_MAX, N_WAVELENGTHS)
    psi_clean, delta_clean = forward_operator(GT_PARAMS, wavelengths, ANGLE_OF_INCIDENCE)

    rng = np.random.default_rng(SEED)
    psi_noisy = psi_clean + NOISE_LEVEL * rng.standard_normal(N_WAVELENGTHS)
    delta_noisy = delta_clean + NOISE_LEVEL * rng.standard_normal(N_WAVELENGTHS)

    print(f"[DATA] Ψ range: [{psi_clean.min():.2f}, {psi_clean.max():.2f}]°")
    print(f"[DATA] Δ range: [{delta_clean.min():.2f}, {delta_clean.max():.2f}]°")
    print(f"[DATA] {N_WAVELENGTHS} wavelengths, θ={ANGLE_OF_INCIDENCE}°")

    return wavelengths, psi_noisy, delta_noisy, psi_clean, delta_clean

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `reconstruct`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. Inverse Solver
# ═══════════════════════════════════════════════════════════
def reconstruct(wavelengths, psi_meas, delta_meas):
    """
    Fit ellipsometric parameters using DE + Nelder-Mead.

    Free parameters: thickness, A, B, C, k_amp

    Returns
    -------
    fit_params : dict
    psi_fit, delta_fit : arrays
    """
    def cost(x):
        thick, A, B, C, k_amp = x
        params = {"thickness": thick, "A": A, "B": B, "C": C, "k_amp": k_amp}
        try:
            psi_calc, delta_calc = forward_operator(params, wavelengths, ANGLE_OF_INCIDENCE)
            res_psi = (psi_meas - psi_calc) / NOISE_LEVEL
            res_delta = (delta_meas - delta_calc) / NOISE_LEVEL
            return np.sum(res_psi**2 + res_delta**2)
        except Exception:
            return 1e20

    bounds = [
        (10, 500),       # thickness [nm]
        (1.3, 2.0),      # A
        (0.0, 0.05),     # B
        (0.0, 0.005),    # C
        (0.0, 0.1),      # k_amp
    ]

    print("[RECON] Stage 1 — Differential Evolution ...")
    result_de = differential_evolution(cost, bounds, seed=SEED, maxiter=150,
                                        tol=1e-5, popsize=15)
    print(f"[RECON]   χ² = {result_de.fun:.2f}")

    print("[RECON] Stage 2 — Nelder-Mead refinement ...")
    result = minimize(cost, result_de.x, method='Nelder-Mead',
                      options={'maxiter': 2000, 'xatol': 1e-6})
    print(f"[RECON]   χ² = {result.fun:.2f}")

    thick, A, B, C, k_amp = result.x
    fit_params = {"thickness": float(thick), "A": float(A), "B": float(B),
                  "C": float(C), "k_amp": float(k_amp)}
    psi_fit, delta_fit = forward_operator(fit_params, wavelengths, ANGLE_OF_INCIDENCE)

    return fit_params, psi_fit, delta_fit

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize_results`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. Metrics
# ═══════════════════════════════════════════════════════════
def compute_metrics(gt, fit, psi_clean, psi_fit, delta_clean, delta_fit, wavelengths):
    """Compute ellipsometry reconstruction metrics."""
    # Ψ metrics
    rmse_psi = float(np.sqrt(np.mean((psi_clean - psi_fit)**2)))
    cc_psi = float(np.corrcoef(psi_clean, psi_fit)[0, 1])

    # Δ metrics
    rmse_delta = float(np.sqrt(np.mean((delta_clean - delta_fit)**2)))
    cc_delta = float(np.corrcoef(delta_clean, delta_fit)[0, 1])

    # Combined PSNR/SSIM on Ψ
    dr = psi_clean.max() - psi_clean.min()
    mse = np.mean((psi_clean - psi_fit)**2)
    psnr = float(10 * np.log10(dr**2 / max(mse, 1e-30)))
    tile_rows = 7
    a2d = np.tile(psi_clean, (tile_rows, 1))
    b2d = np.tile(psi_fit, (tile_rows, 1))
    ssim_val = float(ssim_fn(a2d, b2d, data_range=dr, win_size=7))

    # Parameter recovery
    param_metrics = {}
    for k in ["thickness", "A", "B", "C", "k_amp"]:
        g, f = gt[k], fit[k]
        param_metrics[f"gt_{k}"] = float(g)
        param_metrics[f"fit_{k}"] = float(f)
        param_metrics[f"abs_err_{k}"] = float(abs(g - f))

    # n(λ) recovery
    n_gt = cauchy_n(wavelengths, gt["A"], gt["B"], gt["C"])
    n_fit = cauchy_n(wavelengths, fit["A"], fit["B"], fit["C"])
    cc_n = float(np.corrcoef(n_gt, n_fit)[0, 1])

    return {
        "PSNR_psi": psnr,
        "SSIM_psi": ssim_val,
        "CC_psi": cc_psi,
        "RMSE_psi_deg": rmse_psi,
        "CC_delta": cc_delta,
        "RMSE_delta_deg": rmse_delta,
        "CC_n": cc_n,
        **param_metrics,
    }

# ═══════════════════════════════════════════════════════════
# 6. Visualization
# ═══════════════════════════════════════════════════════════
def visualize_results(wavelengths, psi_meas, delta_meas, psi_clean, delta_clean,
                      psi_fit, delta_fit, gt, fit, metrics, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax = axes[0, 0]
    ax.plot(wavelengths, psi_clean, 'b-', lw=2, label='GT')
    ax.plot(wavelengths, psi_meas, 'k.', ms=2, alpha=0.3, label='Noisy')
    ax.plot(wavelengths, psi_fit, 'r--', lw=1.5, label='Fit')
    ax.set_xlabel('Wavelength [nm]'); ax.set_ylabel('Ψ [°]')
    ax.set_title('(a) Ψ(λ)'); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(wavelengths, delta_clean, 'b-', lw=2, label='GT')
    ax.plot(wavelengths, delta_meas, 'k.', ms=2, alpha=0.3, label='Noisy')
    ax.plot(wavelengths, delta_fit, 'r--', lw=1.5, label='Fit')
    ax.set_xlabel('Wavelength [nm]'); ax.set_ylabel('Δ [°]')
    ax.set_title('(b) Δ(λ)'); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    n_gt = cauchy_n(wavelengths, gt["A"], gt["B"], gt["C"])
    n_fit = cauchy_n(wavelengths, fit["A"], fit["B"], fit["C"])
    ax.plot(wavelengths, n_gt, 'b-', lw=2, label='GT n(λ)')
    ax.plot(wavelengths, n_fit, 'r--', lw=2, label='Fit n(λ)')
    ax.set_xlabel('Wavelength [nm]'); ax.set_ylabel('Refractive index n')
    ax.set_title('(c) Dispersion'); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    labels = ['d [nm]', 'A', 'B', 'C', 'k_amp']
    keys = ['thickness', 'A', 'B', 'C', 'k_amp']
    gt_v = [gt[k] for k in keys]; fit_v = [fit[k] for k in keys]
    x = np.arange(len(labels)); w = 0.35
    ax.bar(x - w/2, gt_v, w, label='GT', color='steelblue')
    ax.bar(x + w/2, fit_v, w, label='Fit', color='tomato')
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_title('(d) Parameters'); ax.legend(); ax.grid(True, alpha=0.3, axis='y')

    fig.suptitle(f"refellips — Spectroscopic Ellipsometry Inversion\n"
                 f"PSNR(Ψ)={metrics['PSNR_psi']:.1f} dB  |  CC(Ψ)={metrics['CC_psi']:.4f}  |  "
                 f"CC(Δ)={metrics['CC_delta']:.4f}  |  Δd={metrics['abs_err_thickness']:.2f} nm",
                 fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"[VIS] Saved → {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("=" * 65)
print("  refellips — Spectroscopic Ellipsometry Inversion")
print("=" * 65)
wavelengths, psi_meas, delta_meas, psi_clean, delta_clean = load_or_generate_data()
fit_params, psi_fit, delta_fit = reconstruct(wavelengths, psi_meas, delta_meas)
metrics = compute_metrics(GT_PARAMS, fit_params, psi_clean, psi_fit,
                          delta_clean, delta_fit, wavelengths)
for k, v in sorted(metrics.items()):
    print(f"  {k:30s} = {v}")
with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), np.column_stack([psi_fit, delta_fit]))
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), np.column_stack([psi_clean, delta_clean]))
visualize_results(wavelengths, psi_meas, delta_meas, psi_clean, delta_clean,
                  psi_fit, delta_fit, GT_PARAMS, fit_params, metrics,
                  os.path.join(RESULTS_DIR, "reconstruction_result.png"))
print("\n" + "=" * 65 + "\n  DONE\n" + "=" * 65)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **refellips**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=N/A (parameter estimation), SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: refnx: neutron and X-ray reflectometry analysis in Python
- Repository: https://github.com/refnx/refnx
